# Square-wave PSTH reference

Runs the E/I SNN with a unit square-wave stimulus (amplitude = 1 for the full
stimulation window) to establish a baseline PSTH against which optimised
waveforms can be compared.  Uses the same reduced-network config as the
optimisation notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from designer_waveform.models import RandomEINetwork, load_config

In [ ]:
cfg = load_config(Path("..") / "configs" / "random_ei_orawe_params.json")
cfg.N_exc    = 2000
cfg.N_inh    = 500
cfg.t_pre_ms = 200.0
cfg.t_post_ms = 100.0

model = RandomEINetwork(cfg)

In [ ]:
# Unit square wave: amplitude 1 for the entire stim window
square_wave = lambda t: np.ones_like(np.asarray(t, dtype=float))

print("Running simulation...")
result = model.run(square_wave)
print("Done.")

psth      = result["psth_exc"]
t_psth_ms = result["t_psth_ms"]
t_stim_ms = result["t_stim_ms"]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

axes[0].axhline(1, color="steelblue", lw=2)
axes[0].set_xlim(t_stim_ms[0], t_stim_ms[-1])
axes[0].set_ylim(-0.1, 1.3)
axes[0].set_xlabel("time in stim window (ms)")
axes[0].set_ylabel("waveform amplitude (a.u.)")
axes[0].set_title("Stimulus: unit square wave")

axes[1].bar(t_psth_ms, psth, width=cfg.psth_bin_ms * 0.9,
            color="steelblue", alpha=0.8)
axes[1].set_xlabel("time in stim window (ms)")
axes[1].set_ylabel("spikes / neuron / bin")
axes[1].set_title("PSTH — excitatory population")

plt.tight_layout()
plt.show()

print(f"Mean firing rate during stim: "
      f"{psth.sum() / (cfg.t_stim_ms / 1000):.1f} spikes / neuron / s")